In [95]:
#Counting the distrubution of antibodies targeting homo sapiens/sars cov 2 antigens in the two clusters

import pandas as pd

df = pd.read_csv("antibodies_pca_clusters.csv")

# Extract antigen species from the first column 
df['antigen_species'] = df.iloc[:, 0].str.extract(r'antigen_species=([\w\s]+)', expand=False)

# Normalize casing and strip spaces 
df['antigen_species'] = df['antigen_species'].str.lower().str.strip()

# Filter only homo sapiens (saved under homo) and sars cov 2 (saved under severe) 
df_filtered = df[df['antigen_species'].isin(['homo', 'severe'])]

# Group by Cluster and antigen_species
counts = df_filtered.groupby(['Cluster', 'antigen_species']).size().unstack(fill_value=0)

print(counts)


antigen_species  homo  severe
Cluster                      
0                 514     882
1                 514     884


In [ ]:
#Chi square test

from scipy.stats import chi2_contingency
import numpy as np

#Contingency table
table = np.array([
    [514, 882],  # Cluster 0
    [514, 884]   # Cluster 1
])

chi2, p, dof, expected = chi2_contingency(table)

print(f"Chi-square statistic: {chi2:.4f}")
print(f"p-value: {p:.4f}")


Chi-square statistic: 0.0000
p-value: 1.0000


The CSV containing the data about CDR3 was missing the data on antigen species needed for the statistical test and also contained a lot of duplicates (Same antibody chain entered twice). The following two chunks fix the file (CDR3_pca_clusters.csv --> CDR3_pca_clusters_final.csv)

In [ ]:
#Annotating the CDR csv (Adding info about the antigen targeted by the antibody)

import pandas as pd

df_cdr3 = pd.read_csv("CDR3_pca_clusters.csv")

#Extract pdb
def extract_pdb(id_str):
    pdb = id_str.split('|')[0].lower().strip()
    return pdb

df_cdr3['pdb'] = df_cdr3.iloc[:, 0].apply(extract_pdb)

#Load metadata file 
df_meta = pd.read_csv("../../data_cleanup/df_sars_hum.csv", delimiter='\t')
df_meta['pdb'] = df_meta['pdb'].str.lower().str.strip()

#Extract pdb and antigen_species columns
df_species = df_meta[['pdb', 'antigen_species']].drop_duplicates()

#Merge only on pdb
df_merged = pd.merge(df_cdr3, df_species, on='pdb', how='left')

#Annotate ID with antigen_species
def update_id_with_species(row):
    original_id = row.iloc[0]
    species = row['antigen_species']
    if pd.isna(species):
        return original_id
    parts = original_id.split('|')
    parts[-1] = species.strip()  # Replace last part with species
    return '|'.join(parts)

df_merged['Annotated_ID'] = df_merged.apply(update_id_with_species, axis=1)

#Drop helper columns
df_merged.drop(columns=['pdb'], inplace=True)


df_merged.to_csv("CDR3_pca_clusters_annotated.csv", index=False)





In [96]:
#Filtering out all the duplicates from the annotated CDR csv

import pandas as pd
import re

df_annotated = pd.read_csv("CDR3_pca_clusters_annotated.csv")

#Extract pdb from annotated IDs
def extract_pdb(id_str):
    pdb = id_str.split('|')[0].lower().strip()
    m = re.match(r'^([a-z0-9]{4})', pdb)
    return m.group(1) if m else pdb

id_col = df_annotated.columns[0]
df_annotated['pdb_clean'] = df_annotated[id_col].apply(extract_pdb)

#Load antibodies metadata to get valid pdbs
df_meta = pd.read_csv("antibodies_pca_clusters.csv")
id_col_meta = df_meta.columns[0]
df_meta['pdb_clean'] = df_meta[id_col_meta].apply(extract_pdb)
valid_pdbs = set(df_meta['pdb_clean'].unique())

#Filter annotated rows by valid pdbs
df_filtered = df_annotated[df_annotated['pdb_clean'].isin(valid_pdbs)].copy()

#Remove duplicates 
df_filtered_no_dupes = df_filtered.drop_duplicates()

#Drop helper column 
df_filtered_no_dupes = df_filtered_no_dupes.drop(columns=['pdb_clean'])

#Save cleaned file
df_filtered_no_dupes.to_csv("CDR3_pca_clusters_final.csv", index=False)




In [ ]:
#Counting the distrubution of antibodies targeting homo sapiens/sars cov 2 antigens in the two clusters

import pandas as pd

df = pd.read_csv("CDR3_pca_clusters_final.csv")

# Split 'antigen_species' column by literal " | ", then explode rows
df_expanded = df.assign(
    antigen_species=df['antigen_species'].str.split(' | ', regex=False)
).explode('antigen_species')

# Normalize casing and strip whitespace
df_expanded['antigen_species'] = df_expanded['antigen_species'].str.lower().str.strip()

# Filter out 'homo sapiens' and 'severe acute respiratory syndrome coronavirus2'
df_filtered = df_expanded[df_expanded['antigen_species'].isin([
    'homo sapiens', 
    'severe acute respiratory syndrome coronavirus2'
])]

# Group by Cluster and antigen_species, count 
counts = df_filtered.groupby(['Cluster', 'antigen_species']).size().unstack(fill_value=0)

print(counts)


antigen_species  homo sapiens  severe acute respiratory syndrome coronavirus2
Cluster                                                                      
0                         752                                            1191
1                         430                                             636


In [94]:
#Chi square test

from scipy.stats import chi2_contingency
import numpy as np

# Construct the contingency table
table = np.array([
    [752, 1191],  # Cluster 0
    [430, 636]   # Cluster 1
])

chi2, p, dof, expected = chi2_contingency(table)

print(f"Chi-square statistic: {chi2:.4f}")
print(f"p-value: {p:.4f}")


Chi-square statistic: 0.7042
p-value: 0.4014
